In [2]:
using PowerModels, DataFrames, CSV, LinearAlgebra, SparseArrays

function extract_opf_constraints(case_path::String, output_dir::String)
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    data = PowerModels.parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    println("Parsed case: $(case_name)")

    df_base_mva = DataFrame(parameter=["baseMVA"], value=[ref[:baseMVA]])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)

    bus_ids = sort(collect(keys(ref[:bus])))
    load_pd = Dict(id => 0.0 for id in bus_ids)
    load_qd = Dict(id => 0.0 for id in bus_ids)

    for (_, load) in ref[:load]
        if get(load, "status", 1) == 1
            b = load["load_bus"]
            load_pd[b] += get(load, "pd", 0.0)
            load_qd[b] += get(load, "qd", 0.0)
        end
    end

    bus_data = DataFrame(
        bus_id   = Int[],
        type     = Int[],
        pd_mw    = Float64[],
        qd_mvar  = Float64[],
        vmin_pu  = Float64[],
        vmax_pu  = Float64[],
        vm_pu    = Float64[],
        va_deg   = Float64[],
        base_kv = Float64[]
    )

    for id in bus_ids
        bus = ref[:bus][id]
        push!(bus_data, [
            id,
            bus["bus_type"],
            load_pd[id],
            load_qd[id],
            bus["vmin"],
            bus["vmax"],
            get(bus, "vm", 1.0),
            get(bus, "va", 0.0),
            get(bus, "base_kv", 110.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_bus_data.csv"), bus_data)

    gen_ids = sort(collect(keys(ref[:gen])))
    gen_data = DataFrame(
        gen_id   = Int[],
        bus_id   = Int[],
        pg_min_mw = Float64[],
        pg_max_mw = Float64[],
        qg_min_mvar = Float64[],
        qg_max_mvar = Float64[],
        vg_pu    = Float64[],
        cost_c2  = Float64[],
        cost_c1  = Float64[],
        cost_c0  = Float64[]
    )

    for id in gen_ids
        g = ref[:gen][id]
        coeffs = get(g, "cost", Float64[])
        c2, c1, c0 = 0.0, 0.0, 0.0
        if length(coeffs) == 3
            c2, c1, c0 = coeffs[1], coeffs[2], coeffs[3]
        elseif length(coeffs) == 2
            c1, c0 = coeffs[1], coeffs[2]
        end

        push!(gen_data, [
            id,
            g["gen_bus"],
            g["pmin"], g["pmax"],
            g["qmin"], g["qmax"],
            g["vg"],
            c2, c1, c0
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_gen_data.csv"), gen_data)

    branch_ids = sort(collect(keys(ref[:branch])))
    branch_data = DataFrame(
        branch_id  = Int[],
        f_bus      = Int[],
        t_bus      = Int[],
        r_pu       = Float64[],
        x_pu       = Float64[],
        b_pu       = Float64[],    
        rate_a_mva = Float64[],      
        tap_ratio  = Float64[],
        shift_deg  = Float64[]
    )

    for id in branch_ids
        br = ref[:branch][id]
        rate_a_raw = get(br, "rate_a", Inf)      
        rate_a_val = isfinite(rate_a_raw) ? rate_a_raw * ref[:baseMVA] : 0.0  

        push!(branch_data, [
            id,
            br["f_bus"], br["t_bus"],
            br["br_r"], br["br_x"],
            get(br, "b_fr", 0.0) + get(br, "b_to", 0.0),
            rate_a_val,
            get(br, "tap", 1.0),
            get(br, "shift", 0.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_branch_data.csv"), branch_data)
    
    slack_buses = DataFrame(bus_id = collect(keys(ref[:ref_buses])))
    CSV.write(joinpath(output_dir, "$(case_name)_slack_buses.csv"), slack_buses)

    println("Done. Files saved to: $(output_dir)")
end

case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\api\pglib_opf_case118_ieee__api.m" 
output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF Constraints\case118(api)"

extract_opf_constraints(case_file, output_dir)


[info | PowerModels]: removing 3 cost terms from generator 32: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 29: [3266.8781000000004, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 1: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 54: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 41: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 51: [3504.3401000000003, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 53: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 27: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 42: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 33: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 28: [3478.1778000000004, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 50: Float64[]
[info

In [6]:
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV
const MOI = MathOptInterface
Random.seed!(42) 

function solve_ACOPF(case_path::String, num_samples::Int=50000, variance::Float64=0.15)
    println("Samples: $(num_samples), Variance: $(variance)")
    
    data = parse_file(case_path)
    
    base_loads_pd = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"] if load["pd"] > 0)
    base_loads_qd = Dict(load["load_bus"] => load["qd"] for (load_idx, load) in data["load"] if haskey(load, "qd"))

    load_bus_indices = sort(collect(keys(base_loads_pd)))
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])
    bus_indices = sort([b["bus_i"] for (i,b) in data["bus"]])

    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    successful_samples = 0
    load_pd_list = Vector{Vector{Float64}}()
    load_qd_list = Vector{Vector{Float64}}() 
    gen_pg_list = Vector{Vector{Float64}}()
    gen_qg_list = Vector{Vector{Float64}}()
    bus_vm_list = Vector{Vector{Float64}}()
    bus_va_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    total_loop_time = @elapsed begin
        @showprogress "Progress" for _ in 1:num_samples 
            temp_data = deepcopy(data)

            new_loads_pd = Dict{Int, Float64}()
            new_loads_qd = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads_pd[bus_idx]
                sigma_pd = abs(base_pd * variance)
                new_loads_pd[bus_idx] = max(0.0, base_pd + sigma_pd * randn())
                
                base_qd = get(base_loads_qd, bus_idx, 0.0)
                sigma_qd = abs(base_qd * variance) 
                new_loads_qd[bus_idx] = base_qd + sigma_qd * randn()
            end

            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(new_loads_pd, bus_idx_int)
                    load_data["pd"] = new_loads_pd[bus_idx_int]
                    load_data["qd"] = new_loads_qd[bus_idx_int] 
                end
            end

            try
                result = solve_ac_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true)))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    push!(cost_list, result["objective"])
                    
                    push!(load_pd_list, [new_loads_pd[i] for i in load_bus_indices])
                    push!(load_qd_list, [new_loads_qd[i] for i in load_bus_indices])
                    
                    push!(gen_pg_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    push!(gen_qg_list, [result["solution"]["gen"][string(i)]["qg"] for i in gen_indices])
                    push!(bus_vm_list, [result["solution"]["bus"][string(i)]["vm"] for i in bus_indices])
                    push!(bus_va_list, [result["solution"]["bus"][string(i)]["va"] for i in bus_indices])
                    
                    successful_samples += 1
                end
            catch e
            end
        end 
    end 
    
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    
    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=2),"", string('$'), "/h")
    end
    
    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2),"ms")
    end
    
    output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case57(v=0.15)" #change the output address
    #output_dir = "/home/ubuntu/acopf_case300(v=0.12)"
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    function save_to_csv(data_list, col_indices, type, name)
        if !isempty(data_list)
            matrix = hcat(data_list...)'
            df = DataFrame(matrix, Symbol.(type .* string.(col_indices)))
            filepath = joinpath(output_dir, "$(case_name)_$(name).csv")
            CSV.write(filepath, df)
        end
    end

    save_to_csv(load_pd_list, load_bus_indices, "pd", "pd")
    save_to_csv(load_qd_list, load_bus_indices, "qd", "qd") 
    save_to_csv(gen_pg_list, gen_indices, "pg_", "pg")
    save_to_csv(gen_qg_list, gen_indices, "qg_", "qg")
    save_to_csv(bus_vm_list, bus_indices, "vm_", "vm")
    save_to_csv(bus_va_list, bus_indices, "va_", "va")
end

# Main
CASE_FILE_PATH = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case57_ieee.m" #change the file path 
#CASE_FILE_PATH = "/home/ubuntu/pglib_opf_case300_ieee.m"
solve_ACOPF(CASE_FILE_PATH, 50000, 0.15)   #change the sample number and variance 

Samples: 50000, Variance: 0.15
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1696.0624, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [3044.1037, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 7: [3718.8979000000004, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [3407.5557000000003, 0.0]


Progress 100%|███████████████████████████████████████████| Time: 1:50:15


Successfully generated 48256 / 50000 samples in 6615.89s.
Average cost: 37584.09$/h
Average solving time for each sample: 137.1ms


"C:\\Users\\Aloha\\Desktop\\dataset\\ACOPF dataset\\case57(v=0.15)\\pglib_opf_case57_ieee_va.csv"

In [7]:
using PowerModels, Ipopt, MathOptInterface, ProgressMeter, DelimitedFiles, Random, Statistics, DataFrames, CSV
const MOI = MathOptInterface
Random.seed!(42) 

function solve_ACOPF(case_path::String, num_samples::Int=50000, variance::Float64=0.10)
    println("Samples: $(num_samples), Variance: $(variance)")
    
    data = parse_file(case_path)
    
    base_loads_pd = Dict(load["load_bus"] => load["pd"] for (load_idx, load) in data["load"] if load["pd"] > 0)
    base_loads_qd = Dict(load["load_bus"] => load["qd"] for (load_idx, load) in data["load"] if haskey(load, "qd"))

    load_bus_indices = sort(collect(keys(base_loads_pd)))
    gen_indices = sort([g["index"] for (i, g) in data["gen"]])
    bus_indices = sort([b["bus_i"] for (i,b) in data["bus"]])

    silent_optimizer = MOI.OptimizerWithAttributes(Ipopt.Optimizer, "print_level" => 0, "sb" => "yes")
    
    successful_samples = 0
    load_pd_list = Vector{Vector{Float64}}()
    load_qd_list = Vector{Vector{Float64}}() 
    gen_pg_list = Vector{Vector{Float64}}()
    gen_qg_list = Vector{Vector{Float64}}()
    bus_vm_list = Vector{Vector{Float64}}()
    bus_va_list = Vector{Vector{Float64}}()
    cost_list = Float64[]

    total_loop_time = @elapsed begin
        @showprogress "Progress" for _ in 1:num_samples 
            temp_data = deepcopy(data)

            new_loads_pd = Dict{Int, Float64}()
            new_loads_qd = Dict{Int, Float64}()
            for bus_idx in load_bus_indices
                base_pd = base_loads_pd[bus_idx]
                sigma_pd = abs(base_pd * variance)
                new_loads_pd[bus_idx] = max(0.0, base_pd + sigma_pd * randn())
                
                base_qd = get(base_loads_qd, bus_idx, 0.0)
                sigma_qd = abs(base_qd * variance) 
                new_loads_qd[bus_idx] = base_qd + sigma_qd * randn()
            end

            for (load_id, load_data) in temp_data["load"]
                bus_idx_int = load_data["load_bus"]
                if haskey(new_loads_pd, bus_idx_int)
                    load_data["pd"] = new_loads_pd[bus_idx_int]
                    load_data["qd"] = new_loads_qd[bus_idx_int] 
                end
            end

            try
                result = solve_ac_opf(temp_data, silent_optimizer; setting = Dict("output" => Dict("branch_flows" => true)))

                if result["termination_status"] == MOI.LOCALLY_SOLVED
                    push!(cost_list, result["objective"])
                    
                    push!(load_pd_list, [new_loads_pd[i] for i in load_bus_indices])
                    push!(load_qd_list, [new_loads_qd[i] for i in load_bus_indices])
                    
                    push!(gen_pg_list, [result["solution"]["gen"][string(i)]["pg"] for i in gen_indices])
                    push!(gen_qg_list, [result["solution"]["gen"][string(i)]["qg"] for i in gen_indices])
                    push!(bus_vm_list, [result["solution"]["bus"][string(i)]["vm"] for i in bus_indices])
                    push!(bus_va_list, [result["solution"]["bus"][string(i)]["va"] for i in bus_indices])
                    
                    successful_samples += 1
                end
            catch e
            end
        end 
    end 
    
    println("Successfully generated $(successful_samples) / $(num_samples) samples in $(round(total_loop_time, digits=2))s.")
    
    if !isempty(cost_list)
        avg_cost = mean(cost_list)
        println("Average cost: ", round(avg_cost, digits=2),"", string('$'), "/h")
    end
    
    if successful_samples > 0
        avg_solve_time_ms = (total_loop_time / successful_samples) * 1000 
        println("Average solving time for each sample: ", round(avg_solve_time_ms, digits=2),"ms")
    end
    
    output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF dataset\case300(v=0.12)" #change the output address
    #output_dir = "/home/ubuntu/acopf_case300(v=0.12)"
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    function save_to_csv(data_list, col_indices, type, name)
        if !isempty(data_list)
            matrix = hcat(data_list...)'
            df = DataFrame(matrix, Symbol.(type .* string.(col_indices)))
            filepath = joinpath(output_dir, "$(case_name)_$(name).csv")
            CSV.write(filepath, df)
        end
    end

    save_to_csv(load_pd_list, load_bus_indices, "pd", "pd")
    save_to_csv(load_qd_list, load_bus_indices, "qd", "qd") 
    save_to_csv(gen_pg_list, gen_indices, "pg_", "pg")
    save_to_csv(gen_qg_list, gen_indices, "qg_", "qg")
    save_to_csv(bus_vm_list, bus_indices, "vm_", "vm")
    save_to_csv(bus_va_list, bus_indices, "va_", "va")
end

# Main
CASE_FILE_PATH = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case300_ieee.m" #change the file path 
#CASE_FILE_PATH = "/home/ubuntu/pglib_opf_case300_ieee.m"
solve_ACOPF(CASE_FILE_PATH, 50000, 0.10)   #change the sample number and variance 

Samples: 50000, Variance: 0.1
[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1696.0624, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [3044.1037, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 7: [3718.8979000000004, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [3407.5557000000003, 0.0]


Progress 100%|███████████████████████████████████████████| Time: 1:21:47


Successfully generated 49840 / 50000 samples in 4907.8s.
Average cost: 37582.27$/h
Average solving time for each sample: 98.47ms


"C:\\Users\\Aloha\\Desktop\\dataset\\ACOPF dataset\\case57(v=0.10)\\pglib_opf_case57_ieee_va.csv"

In [11]:
#new unify units

using PowerModels, DataFrames, CSV, LinearAlgebra, SparseArrays

"""
提取OPF约束数据，所有功率和成本单位均保存为 p.u.（标幺值）。
"""
function extract_opf_constraints_pu(case_path::String, output_dir::String)
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    data = PowerModels.parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    println("Parsed case: $(case_name)")

    # 获取 baseMVA 并保存 (它定义了p.u.系统的基准)
    base_mva = ref[:baseMVA]
    df_base_mva = DataFrame(parameter=["baseMVA"], value=[base_mva])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)

    # --- 母线数据 ---
    bus_ids = sort(collect(keys(ref[:bus])))
    
    # 聚合负荷 (这些值已经是 p.u.)
    load_pd = Dict(id => 0.0 for id in bus_ids)
    load_qd = Dict(id => 0.0 for id in bus_ids)

    for (_, load) in ref[:load]
        if get(load, "status", 1) == 1
            b = load["load_bus"]
            load_pd[b] += get(load, "pd", 0.0) # p.u.
            load_qd[b] += get(load, "qd", 0.0) # p.u.
        end
    end

    bus_data = DataFrame(
        bus_id  = Int[],
        type    = Int[],
        pd_pu   = Float64[], # <-- 改为 p.u.
        qd_pu   = Float64[], # <-- 改为 p.u.
        vmin_pu = Float64[],
        vmax_pu = Float64[],
        vm_pu   = Float64[],
        va_deg  = Float64[],
        base_kv = Float64[]
    )

    for id in bus_ids
        bus = ref[:bus][id]
        push!(bus_data, [
            id,
            bus["bus_type"],
            load_pd[id],  # <-- 直接保存 p.u. 值
            load_qd[id],  # <-- 直接保存 p.u. 值
            bus["vmin"],
            bus["vmax"],
            get(bus, "vm", 1.0),
            get(bus, "va", 0.0),
            get(bus, "base_kv", 110.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_bus_data.csv"), bus_data)

    # --- 发电机数据 ---
    gen_ids = sort(collect(keys(ref[:gen])))
    gen_data = DataFrame(
        gen_id      = Int[],
        bus_id      = Int[],
        pg_min_pu   = Float64[], # <-- 改为 p.u.
        pg_max_pu   = Float64[], # <-- 改为 p.u.
        qg_min_pu   = Float64[], # <-- 改为 p.u.
        qg_max_pu   = Float64[], # <-- 改为 p.u.
        vg_pu       = Float64[],
        cost_c2     = Float64[], # <-- 对应 p.u. 输入的成本系数
        cost_c1     = Float64[], # <-- 对应 p.u. 输入的成本系数
        cost_c0     = Float64[]
    )

    for id in gen_ids
        g = ref[:gen][id]
        
        # 获取 p.u. 成本系数
        coeffs = get(g, "cost", Float64[])
        c2, c1, c0 = 0.0, 0.0, 0.0
        if length(coeffs) == 3
            c2, c1, c0 = coeffs[1], coeffs[2], coeffs[3]
        elseif length(coeffs) == 2
            c1, c0 = coeffs[1], coeffs[2]
        end

        push!(gen_data, [
            id,
            g["gen_bus"],
            g["pmin"], # <-- 直接保存 p.u. 值
            g["pmax"], # <-- 直接保存 p.u. 值
            g["qmin"], # <-- 直接保存 p.u. 值
            g["qmax"], # <-- 直接保存 p.u. 值
            g["vg"],
            c2, c1, c0 # <-- 直接保存 p.u. 对应的成本系数
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_gen_data.csv"), gen_data)

    # --- 支路数据 ---
    branch_ids = sort(collect(keys(ref[:branch])))
    branch_data = DataFrame(
        branch_id = Int[],
        f_bus     = Int[],
        t_bus     = Int[],
        r_pu      = Float64[],
        x_pu      = Float64[],
        b_pu      = Float64[],   
        rate_a_pu = Float64[], # <-- 改为 p.u.
        tap_ratio = Float64[],
        shift_deg = Float64[]
    )

    for id in branch_ids
        br = ref[:branch][id]
        
        # 获取 p.u. 额定容量
        rate_a_raw = get(br, "rate_a", Inf)     
        # 如果是 Inf 或 0 (无限制)，则保存为 0.0，否则保存 p.u. 值
        rate_a_val = isfinite(rate_a_raw) ? rate_a_raw : 0.0 

        push!(branch_data, [
            id,
            br["f_bus"], br["t_bus"],
            br["br_r"], br["br_x"],
            get(br, "b_fr", 0.0) + get(br, "b_to", 0.0), # 总对地导纳
            rate_a_val, # <-- 直接保存 p.u. 值 (或 0)
            get(br, "tap", 1.0),
            get(br, "shift", 0.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_branch_data.csv"), branch_data)
    
    # --- 备用母线数据 ---
    slack_buses = DataFrame(bus_id = collect(keys(ref[:ref_buses])))
    CSV.write(joinpath(output_dir, "$(case_name)_slack_buses.csv"), slack_buses)

    println("Done (p.u. units). Files saved to: $(output_dir)")
end

# --- 使用示例 ---
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case118_ieee.m"
output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF Constraints\case118"

# 注意：调用函数名已改为 extract_opf_constraints_pu
extract_opf_constraints_pu(case_file, output_dir)

[info | PowerModels]: removing 3 cost terms from generator 32: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 29: [3266.8781000000004, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 1: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 54: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 41: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 51: [3504.3401000000003, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 53: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 27: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 42: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 33: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 28: [3478.1778000000004, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 50: Float64[]
[info

In [7]:
#new unify units

using PowerModels, DataFrames, CSV, LinearAlgebra, SparseArrays

"""
提取OPF约束数据，所有功率和成本单位均保存为 p.u.（标幺值）。
"""
function extract_opf_constraints_pu(case_path::String, output_dir::String)
    mkpath(output_dir)
    case_name = split(basename(case_path), ".")[1]

    data = PowerModels.parse_file(case_path)
    ref = PowerModels.build_ref(data)[:it][:pm][:nw][0]

    println("Parsed case: $(case_name)")

    # 获取 baseMVA 并保存 (它定义了p.u.系统的基准)
    base_mva = ref[:baseMVA]
    df_base_mva = DataFrame(parameter=["baseMVA"], value=[base_mva])
    CSV.write(joinpath(output_dir, "$(case_name)_base_mva.csv"), df_base_mva)

    # --- 母线数据 ---
    bus_ids = sort(collect(keys(ref[:bus])))
    
    # 聚合负荷 (这些值已经是 p.u.)
    load_pd = Dict(id => 0.0 for id in bus_ids)
    load_qd = Dict(id => 0.0 for id in bus_ids)

    for (_, load) in ref[:load]
        if get(load, "status", 1) == 1
            b = load["load_bus"]
            load_pd[b] += get(load, "pd", 0.0) # p.u.
            load_qd[b] += get(load, "qd", 0.0) # p.u.
        end
    end

    bus_data = DataFrame(
        bus_id  = Int[],
        type    = Int[],
        pd_pu   = Float64[], # <-- 改为 p.u.
        qd_pu   = Float64[], # <-- 改为 p.u.
        vmin_pu = Float64[],
        vmax_pu = Float64[],
        vm_pu   = Float64[],
        va_deg  = Float64[],
        base_kv = Float64[]
    )

    for id in bus_ids
        bus = ref[:bus][id]
        push!(bus_data, [
            id,
            bus["bus_type"],
            load_pd[id],  # <-- 直接保存 p.u. 值
            load_qd[id],  # <-- 直接保存 p.u. 值
            bus["vmin"],
            bus["vmax"],
            get(bus, "vm", 1.0),
            get(bus, "va", 0.0),
            get(bus, "base_kv", 110.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_bus_data.csv"), bus_data)

    # --- 发电机数据 ---
    gen_ids = sort(collect(keys(ref[:gen])))
    gen_data = DataFrame(
        gen_id      = Int[],
        bus_id      = Int[],
        pg_min_pu   = Float64[], # <-- 改为 p.u.
        pg_max_pu   = Float64[], # <-- 改为 p.u.
        qg_min_pu   = Float64[], # <-- 改为 p.u.
        qg_max_pu   = Float64[], # <-- 改为 p.u.
        vg_pu       = Float64[],
        cost_c2     = Float64[], # <-- 对应 p.u. 输入的成本系数
        cost_c1     = Float64[], # <-- 对应 p.u. 输入的成本系数
        cost_c0     = Float64[]
    )

    for id in gen_ids
        g = ref[:gen][id]
        
        # 获取 p.u. 成本系数
        coeffs = get(g, "cost", Float64[])
        c2, c1, c0 = 0.0, 0.0, 0.0
        if length(coeffs) == 3
            c2, c1, c0 = coeffs[1], coeffs[2], coeffs[3]
        elseif length(coeffs) == 2
            c1, c0 = coeffs[1], coeffs[2]
        end

        push!(gen_data, [
            id,
            g["gen_bus"],
            g["pmin"], # <-- 直接保存 p.u. 值
            g["pmax"], # <-- 直接保存 p.u. 值
            g["qmin"], # <-- 直接保存 p.u. 值
            g["qmax"], # <-- 直接保存 p.u. 值
            g["vg"],
            c2, c1, c0 # <-- 直接保存 p.u. 对应的成本系数
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_gen_data.csv"), gen_data)

    # --- 母线-发电机映射 ---
    n_buses = length(bus_ids)
    n_gen = length(gen_ids)
    bus_id_to_idx = Dict(bus_id => idx for (idx, bus_id) in enumerate(bus_ids))
    
    bus_gen_matrix = zeros(Int, n_buses, n_gen)
    for (gen_idx, gen_id) in enumerate(gen_ids)
        gen_bus_id = ref[:gen][gen_id]["gen_bus"]
        bus_idx = bus_id_to_idx[gen_bus_id]
        bus_gen_matrix[bus_idx, gen_idx] = 1
    end
    
    gen_col_names = ["gen_$i" for i in 1:n_gen]
    bus_gen_df = DataFrame(bus_gen_matrix, gen_col_names)
    insertcols!(bus_gen_df, 1, :bus_id => bus_ids)
    CSV.write(joinpath(output_dir, "$(case_name)_bus_gen_map.csv"), bus_gen_df)

    # --- 支路数据 ---
    branch_ids = sort(collect(keys(ref[:branch])))
    branch_data = DataFrame(
        branch_id = Int[],
        f_bus     = Int[],
        t_bus     = Int[],
        r_pu      = Float64[],
        x_pu      = Float64[],
        b_pu      = Float64[],   
        rate_a_pu = Float64[], # <-- 改为 p.u.
        tap_ratio = Float64[],
        shift_deg = Float64[]
    )

    for id in branch_ids
        br = ref[:branch][id]
        
        # 获取 p.u. 额定容量
        rate_a_raw = get(br, "rate_a", Inf)     
        # 如果是 Inf 或 0 (无限制)，则保存为 0.0，否则保存 p.u. 值
        rate_a_val = isfinite(rate_a_raw) ? rate_a_raw : 0.0 

        push!(branch_data, [
            id,
            br["f_bus"], br["t_bus"],
            br["br_r"], br["br_x"],
            get(br, "b_fr", 0.0) + get(br, "b_to", 0.0), # 总对地导纳
            rate_a_val, # <-- 直接保存 p.u. 值 (或 0)
            get(br, "tap", 1.0),
            get(br, "shift", 0.0)
        ])
    end
    CSV.write(joinpath(output_dir, "$(case_name)_branch_data.csv"), branch_data)
    
    # --- 备用母线数据 ---
    slack_buses = DataFrame(bus_id = collect(keys(ref[:ref_buses])))
    CSV.write(joinpath(output_dir, "$(case_name)_slack_buses.csv"), slack_buses)

    println("Done (p.u. units). Files saved to: $(output_dir)")
end

# --- 使用示例 ---
case_file = raw"C:\Users\Aloha\Desktop\dataset\PGlib\standard\pglib_opf_case57_ieee.m"
output_dir = raw"C:\Users\Aloha\Desktop\dataset\ACOPF Constraints\case57"

# 注意：调用函数名已改为 extract_opf_constraints_pu
extract_opf_constraints_pu(case_file, output_dir)

[info | PowerModels]: removing 3 cost terms from generator 4: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 1: [1696.0624, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 5: [3044.1037, 0.0]
[info | PowerModels]: removing 3 cost terms from generator 2: Float64[]
[info | PowerModels]: removing 3 cost terms from generator 6: Float64[]
[info | PowerModels]: removing 1 cost terms from generator 7: [3718.8979000000004, 0.0]
[info | PowerModels]: removing 1 cost terms from generator 3: [3407.5557000000003, 0.0]
Parsed case: pglib_opf_case57_ieee
Done (p.u. units). Files saved to: C:\Users\Aloha\Desktop\dataset\ACOPF Constraints\case57
